In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

C:\Users\rahul\AppData\Local\Temp\ipykernel_15040\615555437.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [3]:
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

Sample document create in : C:\Users\rahul\AppData\Local\Temp\tmpnx07quso


In [4]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)

In [5]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

loader=DirectoryLoader(
    "data",
    glob="**.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")

Loaded 3 documents

First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [6]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natur

In [7]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

chunks=text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'data\\doc_0.txt'}


In [8]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

In [9]:
sample_text="MAchine LEarning is fascinating"

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

vector = embeddings.embed_query(sample_text)
vector

d:\RAG_Learnings\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1695.33it/s]


[-0.04763007536530495,
 -0.0952337458729744,
 0.08622036874294281,
 0.008249829523265362,
 0.012193533591926098,
 -0.050495218485593796,
 -0.007320770062506199,
 -0.05885446444153786,
 -0.03901907056570053,
 -0.02866092510521412,
 -0.08570096641778946,
 0.07921122014522552,
 0.004685720894485712,
 -0.021505016833543777,
 -0.07088703662157059,
 0.014649470336735249,
 -0.013478705659508705,
 -0.015381776727735996,
 -0.08215580135583878,
 -0.12226205319166183,
 -0.0034172271843999624,
 0.013044707477092743,
 -0.008553915657103062,
 0.00796373188495636,
 0.024985069409012794,
 0.018824247643351555,
 0.06268899142742157,
 0.019554194062948227,
 0.046845268458127975,
 -0.06655009835958481,
 0.0034160232171416283,
 0.06802241504192352,
 -0.011118374764919281,
 0.03004433587193489,
 -0.09648729115724564,
 0.02735750377178192,
 0.009005087427794933,
 0.017103223130106926,
 0.04554038867354393,
 0.00637142825871706,
 -0.005477529019117355,
 -0.04323987290263176,
 0.036371972411870956,
 -0.004953

In [11]:
text_chunks=[]

for i in chunks:
    text_chunks.append(i.page_content)

vector1 = embeddings.embed_documents(text_chunks)
vector1

[[-0.015595534816384315,
  -0.06318730860948563,
  0.04429002106189728,
  0.002251754980534315,
  0.04224461317062378,
  0.019965311512351036,
  -0.03364090994000435,
  -0.06727663427591324,
  -0.04835936799645424,
  0.013172450475394726,
  -0.06509248912334442,
  0.016410665586590767,
  0.026624154299497604,
  -0.004580572713166475,
  -0.023316094651818275,
  -0.018733058124780655,
  0.02075757458806038,
  -0.0047962781973183155,
  -0.02997085452079773,
  -0.06975759565830231,
  0.061650294810533524,
  0.00406449381262064,
  -0.030921299010515213,
  0.017059698700904846,
  -0.015623552724719048,
  0.047161635011434555,
  0.08883307129144669,
  0.07057081907987595,
  0.03914608433842659,
  -0.04054269194602966,
  0.03576301783323288,
  -0.05056978762149811,
  0.02518901601433754,
  0.024980369955301285,
  -0.07496912032365799,
  -0.02607877366244793,
  -0.04071006923913956,
  0.007767923641949892,
  0.005976446438580751,
  0.014141849242150784,
  -0.015410088002681732,
  -0.05404654517

In [12]:
persist_directory ="./chroma_db"

embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1335.99it/s]


Vector store created with 15 vectors
Persisted to: ./chroma_db


In [13]:
query="What are the types of machine learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, w

In [14]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
  0.5356124043464661),
 (Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labele

In [15]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

'''llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False
)

chat_model = ChatHuggingFace(llm=llm)

test_response =chat_model.invoke("What is Large Language Models")
test_response'''

'llm = HuggingFaceEndpoint(\n    repo_id="HuggingFaceH4/zephyr-7b-beta",\n    task="text-generation",\n    max_new_tokens=512,\n    do_sample=False\n)\n\nchat_model = ChatHuggingFace(llm=llm)\n\ntest_response =chat_model.invoke("What is Large Language Models")\ntest_response'

In [16]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1243.85it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [17]:
import os

print(os.getcwd())

d:\RAG_Learnings\2-VectorStore&VectorDatabase


In [18]:
import os

print(os.path.exists(r"D:\RAG_Learnings\.env"))

True


In [19]:
import os

print("Location 1:", os.path.exists(r"D:\RAG_Learnings\.env"))

print("Location 2:", os.path.exists(
    r"D:\RAG_Learnings\2-VectorStore&VectorDatabase\.env"
))

Location 1: True
Location 2: False


In [32]:
'''from dotenv import load_dotenv
import os

load_dotenv(r"D:\RAG_Learnings\.env", override=True)

groq_api_key = os.getenv("GROQ_API_KEY")

print("API Key loaded:", bool(groq_api_key))'''

<>:4: SyntaxWarning: invalid escape sequence '\R'
<>:4: SyntaxWarning: invalid escape sequence '\R'
C:\Users\rahul\AppData\Local\Temp\ipykernel_15040\3079969350.py:4: SyntaxWarning: invalid escape sequence '\R'
  load_dotenv(r"D:\RAG_Learnings\.env", override=True)


'from dotenv import load_dotenv\nimport os\n\nload_dotenv(r"D:\\RAG_Learnings\\.env", override=True)\n\ngroq_api_key = os.getenv("GROQ_API_KEY")\n\nprint("API Key loaded:", bool(groq_api_key))'

In [21]:
'''import os
from dotenv import load_dotenv

load_dotenv(override=True)

key = os.getenv("GROQ_API_KEY")
print(repr(key))'''

'import os\nfrom dotenv import load_dotenv\n\nload_dotenv(override=True)\n\nkey = os.getenv("GROQ_API_KEY")\nprint(repr(key))'

In [22]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=512,
)

print(llm.invoke("What are Large Language Models?").content)

**Large Language Models (LLMs)** are a class of artificial‑intelligence systems that learn to understand, generate, and manipulate human language by training on massive amounts of text data. They are called “large” because they contain millions to billions of trainable parameters (weights) that encode statistical patterns in language.

---

## 1. Core Architecture

| Feature | What it is | Why it matters |
|---------|------------|----------------|
| **Transformer** | The neural‑network backbone introduced in 2017 (Vaswani et al.) | Uses self‑attention to capture long‑range dependencies without recurrence, enabling efficient parallel training. |
| **Self‑Attention** | Mechanism that lets each token weigh every other token in the sequence | Allows the model to focus on relevant context, crucial for understanding nuance. |
| **Layer Normalization & Residual Connections** | Stabilize training and help gradients flow | Enable very deep networks (tens to hundreds of layers). |
| **Positional

In [23]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025C3F3E7390>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025C3F3E7D90>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [24]:
llm.invoke("What is AI")

AIMessage(content='**Artificial Intelligence (AI)** is the field of computer science that focuses on creating systems or machines that can perform tasks that normally require human intelligence. These tasks include reasoning, learning, perception, understanding natural language, recognizing patterns, and making decisions.\n\n---\n\n### Core Concepts\n\n| Concept | What it means | Example |\n|---------|---------------|---------|\n| **Intelligence** | The ability to adapt, solve problems, and learn from experience | A robot learning to navigate a maze |\n| **Machine Learning** | Algorithms that improve performance by learning from data | A spam filter that gets better at detecting junk emails |\n| **Deep Learning** | A subset of ML that uses layered neural networks to model complex patterns | Image recognition that distinguishes cats from dogs |\n| **Narrow (Weak) AI** | Systems designed for a specific task | Voice assistants (Siri, Alexa) |\n| **General (Strong) AI** | Systems that can 

In [25]:
from langchain_classic.chains import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain
)
from langchain_classic.chains import create_retrieval_chain

### STEP 1:Convert vector store to retriever

In [26]:
retriever = vectorstore.as_retriever(search_kwarg={"k":3})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000025C2FBEBB60>, search_kwargs={})

### STEP 2:Create a prompt template

In [27]:
system_prompt ="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context:{context}"""

prompt=ChatPromptTemplate([("system",system_prompt),("human","{input}")])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext:{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

### STEP3:create_stuff_documents_chain

In [28]:
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext:{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 20B', 'release_date'

#### STEP4:create_retrieval_chain

In [29]:
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000025C2FBEBB60>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you

### STEP5:Test your RAG Pipeline

In [30]:
response=rag_chain.invoke({"input":"What is Deep LEarning"})
response

{'input': 'What is Deep LEarning',
 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    process

### End-to-End RAG Pipeline: Sequential Execution and Detailed Walkthrough of Steps 1–5

In [31]:
# =====================================================================
# RAG PIPELINE - FULLY COMMENTED VERSION (written for complete beginners)
#
# RAG = Retrieval-Augmented Generation.
# Plain-English meaning: before an AI answers your question, it first
# LOOKS UP relevant notes from your own documents, then writes the answer
# using those notes. Think of an "open-book exam" instead of answering
# purely from memory.
#
# IMPORTANT: this code assumes two things already exist from earlier
# cells in your notebook:
#   1. `vectorstore` -> a database holding your documents, already split
#                       into small chunks and converted to numbers
#                       (called "embeddings") so they can be searched.
#   2. `llm`         -> the language model (the "brain" that writes the
#                       final answer), e.g. a ChatGPT / Gemini / Llama model.
# Without those two, the code below will give a "name not defined" error.
# =====================================================================


# ---------------------------------------------------------------------
# IMPORTS = borrowing ready-made tools written by other programmers
# ---------------------------------------------------------------------

# Borrow the tool that builds the FINAL machine (Step 4) which joins
# "searching the documents" + "answering the question" into one pipeline.
from langchain_classic.chains import create_retrieval_chain

# Borrow the tool that lets us write a fill-in-the-blanks message
# template to give instructions to the AI (used in Step 2).
from langchain_core.prompts import ChatPromptTemplate

# Borrow the tool that builds the "answer-writing" machine (Step 3).
# It takes the found document pieces, STUFFS them all into one message,
# and hands that message to the AI.
from langchain_classic.chains.combine_documents import (create_stuff_documents_chain)

# NOTE: this is the SAME import as the first line, written a second time.
# It is harmless (Python just ignores the repeat) but you can delete it.
from langchain_classic.chains import create_retrieval_chain


# ---------------------------------------------------------------------
# STEP 1: Convert the vector store into a "retriever"
# ---------------------------------------------------------------------
# A vector store is like a giant library shelf of your document pieces.
# A retriever is the LIBRARIAN: you give it a question and it walks to the
# shelf and brings back the most relevant pieces.
#
# `vectorstore.as_retriever(...)` = "turn this shelf into a librarian".
# `search_kwargs={"k": 3}`        = settings for the librarian.
#                                   "k": 3 means "bring back the 3 best-
#                                   matching pieces" (not 1, not 10).
#
# FIX MADE: your original had `search_kwarg` (missing the final "s").
# The correct name is `search_kwargs`. With the typo the "k=3" setting is
# NOT applied properly (it is ignored or errors, depending on version),
# and the retriever falls back to its default number of results (4).
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Typing the variable name alone on the last line of a notebook cell just
# PRINTS it, so you can see the object was created.
retriever


# ---------------------------------------------------------------------
# STEP 2: Create a prompt template
# ---------------------------------------------------------------------
# A prompt is the written instruction we give to the AI.
# A TEMPLATE is a prompt with blanks ({...}) that get filled in later,
# like a form letter: "Dear {name}, ..."

# The text below is the rule-book the AI must follow.
# It is written in triple quotes """ ... """ so it can span several lines.
# (Comments cannot go INSIDE the quotes, because everything inside the
#  quotes is treated as real text sent to the AI. So they are explained here.)
#   Line 1 -> tells the AI what job it has: answering questions.
#   Line 2 -> tells the AI to base its answer on the retrieved notes.
#   Line 3 -> "If you don't know, say so" stops the AI from making things up.
#   Line 4 -> "Three sentences maximum" keeps answers short.
#   Line 5 -> "Context:{context}" is a BLANK. Later, LangChain automatically
#             pastes the 3 retrieved document pieces in place of {context}.
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Context:{context}"""

# Build the actual template from a list of "messages":
#   ("system", system_prompt) -> the rule-book message. "system" means
#                                "behind-the-scenes instructions to the AI".
#   ("human", "{input}")      -> the user's message. {input} is a second
#                                BLANK that will be filled with your question.
prompt = ChatPromptTemplate([("system", system_prompt), ("human", "{input}")])

# Print the template so you can inspect it.
prompt


# ---------------------------------------------------------------------
# STEP 3: create_stuff_documents_chain
# ---------------------------------------------------------------------
# A "chain" = a pipeline of steps run one after another.
# This chain does: take the retrieved pieces -> "stuff" them into the
# {context} blank -> fill {input} with the question -> send the finished
# message to the AI -> return the AI's answer.
#
# It needs two ingredients:
#   llm    -> the AI brain that will write the answer
#   prompt -> the template (from Step 2) that tells it how to behave
document_chain = create_stuff_documents_chain(llm, prompt)

# Print the chain to inspect it.
document_chain


# ---------------------------------------------------------------------
# STEP 4: create_retrieval_chain
# ---------------------------------------------------------------------
# Now we join the two halves into ONE complete machine:
#   retriever      -> the librarian (finds the 3 relevant pieces)
#   document_chain -> the writer (reads those pieces and answers)
# Final flow: question -> librarian finds notes -> writer answers.
rag_chain = create_retrieval_chain(retriever, document_chain)

# Print the finished pipeline to inspect it.
rag_chain


# ---------------------------------------------------------------------
# STEP 5: Test your RAG pipeline
# ---------------------------------------------------------------------
# `.invoke(...)` means "run the machine once".
# We hand it a dictionary (a labelled box). The label "input" MUST match
# the {input} blank in our prompt template from Step 2.
# (Spelling note: "LEarning" has a stray capital E; harmless here, since
#  the search works on meaning rather than exact spelling.)
response = rag_chain.invoke({"input": "What is Deep LEarning"})

# Show the result. It is a dictionary with three labelled items:
#   "input"   -> your original question
#   "context" -> the 3 document pieces the librarian found
#   "answer"  -> the AI's final written answer (the part you usually want)
response

# To print only the answer, use:  print(response["answer"])

{'input': 'What is Deep LEarning',
 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    process

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [33]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

### STEP 1: Create a custom prompt

In [34]:
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

### STEP 2: create_retrieval_chain

In [36]:
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000025C2FBEBB60>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf

### STEP 3:Document Formatting and Consolidation(i.e. Format and Combine Retrieved Documents into Context)

In [37]:
def format_docs(docs):
    formatted_text = ""

    for doc in docs:
        formatted_text = formatted_text+doc.page_content+"\n\n"
    return formatted_text

### STEP 4:Build the chain ussing LCEL

In [38]:
rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000025C2FBEBB60>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_

### STEP5:Test your RAG Pipeline

In [42]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep learning is a branch of machine learning that uses artificial neural networks composed of many layers of interconnected nodes. These networks are inspired by the structure and function of the human brain. By stacking multiple layers, deep learning models can automatically learn hierarchical representations of data, which has led to major breakthroughs in areas such as computer vision, natural language processing, and speech recognition. For example, Convolutional Neural Networks (CNNs) excel at image processing, while Recurrent Neural Networks (RNNs) and Transformer architectures are particularly effective for sequential data like text and audio.'

In [45]:
retriever.invoke("What is Deep Learning")

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neura

### End-to-End RAG Pipeline Using LCEL: Sequential Execution and Detailed Walkthrough of Steps 1–5

In [47]:
# Import the prompt-template class.
# It lets us create a reusable prompt with {context} and {question} placeholders.
from langchain_core.prompts import ChatPromptTemplate

# Import a runnable that passes its input through without changing it.
from langchain_core.runnables import RunnablePassthrough

# Import a parser that converts the model result into plain text.
from langchain_core.output_parsers import StrOutputParser


# STEP 1: Create a prompt template.
# {context} will later be replaced with the text retrieved from your documents.
# {question} will later be replaced with the user's original question.
custom_prompt = ChatPromptTemplate.from_template(
    """Use the following context to answer the question.

If you don't know the answer based on the context, say you don't know.

Provide specific details from the context to support your answer.

Context:

{context}

Question: {question}

Answer:"""
)


# STEP 3: Define a function to combine the retrieved documents.
def format_docs(docs):
    # Start with an empty string. We will add each document's text to it.
    formatted_text = ""

    # Visit each Document object returned by the retriever, one by one.
    for doc in docs:
        # Add its page_content (main text), followed by a blank line.
        formatted_text = formatted_text + doc.page_content + "\n\n"

    # Give the combined text back to the chain.
    return formatted_text


# STEP 4: Build the RAG pipeline with LCEL.
rag_chain_lcel = (
    {
        # Branch 1: search for relevant documents, then combine their text.
        # The result becomes the value supplied to the prompt's {context}.
        "context": retriever | format_docs,

        # Branch 2: keep the original question unchanged.
        # The result becomes the value supplied to the prompt's {question}.
        "question": RunnablePassthrough()
    }

    # Fill the prompt placeholders using the context and question values.
    | custom_prompt

    # Send the completed prompt to the language model to generate an answer.
    | llm

    # Convert the model output into a plain text string.
    | StrOutputParser()
)


# STEP 5: Ask a question and run the complete RAG pipeline.
response = rag_chain_lcel.invoke("What is Deep Learning")

# Display the final answer.
print(response)


# OPTIONAL: Run only the retriever to inspect the source documents.
docs = retriever.invoke("What is Deep Learning")

# Display the retrieved Document objects (text and metadata).
print(docs)


Deep learning is a subset of machine learning that uses artificial neural networks.  
These networks are inspired by the human brain and are composed of layers of interconnected nodes.  
According to the context, deep learning has transformed fields such as computer vision, natural‑language processing, and speech recognition. It also highlights specific architectures—Convolutional Neural Networks (CNNs) for image processing and Recurrent Neural Networks (RNNs) and Transformers for sequential data.
[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image pr

### Add New Documents To Existing Vector Store

In [48]:
vectorstore

In [49]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [50]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [51]:
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [52]:
vectorstore.add_documents(new_chunks)

['b7eff336-2ae2-49f7-a0f8-a6b4184fec07',
 'bbd345ef-06b9-4533-a858-a078dda6a6c3']

In [53]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 2 new chunks to the vector store
Total vectors now: 17


In [56]:
new_question="What are the keys concepts in reinforcement learning"
result=rag_chain_lcel.invoke(new_question)
result

'The key concepts in reinforcement learning are:\n\n1. **States** – The representations of the environment at a given time that the agent observes.  \n2. **Actions** – The set of possible moves or decisions the agent can take in each state.  \n3. **Rewards** – The numerical feedback (positive or negative) the agent receives after taking an action, guiding learning toward desirable behavior.  \n4. **Policies** – The strategy or mapping from states to actions that the agent follows, which it seeks to optimize.  \n5. **Value Functions** – Functions that estimate the expected cumulative reward from a state (or state-action pair), helping the agent evaluate how good a situation or action is.  \n\nThese concepts form the core framework that enables an RL agent to learn by interacting with its environment and maximizing cumulative reward over time.'

In [58]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [59]:
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt =ChatPromptTemplate.from_messages([
    ("system",contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

In [60]:
history_aware_retriever = create_history_aware_retriever(
    llm,retriever,contextualize_q_prompt
)

history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000025C2FBEBB60>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, T

In [61]:
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

In [62]:
# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,question_answer_chain
)

print("Conversational RAG chain created!")

Conversational RAG chain created!


In [63]:
chat_history =[]
result1 =conversational_rag_chain.invoke({
    "chat_history":chat_history,
    "input":"what is machine learning"
})

print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a branch of artificial intelligence that lets systems learn and improve from experience without being explicitly programmed. It typically uses three main approaches: supervised learning (with labeled data), unsupervised learning (discovering patterns in unlabeled data), and reinforcement learning (learning through trial and error).


In [65]:
chat_history.extend([
    HumanMessage(content="what is machine learning"),
    AIMessage(content=result1["answer"])
])

chat_history

[HumanMessage(content='what is machine learning', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a branch of artificial intelligence that lets systems learn and improve from experience without being explicitly programmed. It typically uses three main approaches: supervised learning (with labeled data), unsupervised learning (discovering patterns in unlabeled data), and reinforcement learning (learning through trial and error).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what is machine learning', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a branch of artificial intelligence that lets systems learn and improve from experience without being explicitly programmed. It typically uses three main approaches: supervised learning (with labeled data), unsupervised learning (discovering patterns in unlabeled data), and reinforcement learning (learning thr

In [66]:
## Follow up question
# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

{'chat_history': [HumanMessage(content='what is machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a branch of artificial intelligence that lets systems learn and improve from experience without being explicitly programmed. It typically uses three main approaches: supervised learning (with labeled data), unsupervised learning (discovering patterns in unlabeled data), and reinforcement learning (learning through trial and error).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='what is machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a branch of artificial intelligence that lets systems learn and improve from experience without being explicitly programmed. It typically uses three main approaches: supervised learning (with labeled data), unsupervised learning (discovering patterns in unlabeled data), and reinforcement le

In [67]:
result2['answer']

'The three main types of machine learning are:\n\n1. **Supervised learning** – models are trained on labeled data.  \n2. **Unsupervised learning** – models discover patterns in unlabeled data.  \n3. **Reinforcement learning** – models learn by receiving rewards or penalties from interactions with an environment.'